In [38]:
from langgraph.graph import StateGraph, START, END
from langchain_huggingface import ChatHuggingFace,HuggingFaceEndpoint
from typing import TypedDict, Optional
from dotenv import load_dotenv

In [39]:
load_dotenv()

True

In [40]:
import os
hf_token = os.getenv("HUGGINGFACE_API_TOKEN")

In [46]:
llm = HuggingFaceEndpoint(
    repo_id = "Qwen/Qwen3-4B-Instruct-2507",
    huggingfacehub_api_token=hf_token,
    temperature = 0.7,
    max_new_tokens=216
)
model = ChatHuggingFace(llm=llm)

In [42]:
# Create a state
class LLMState(TypedDict):
    question: str
    answer: str

In [43]:
def llm_qa(state: LLMState)->LLMState:
    # Extract the question from state
    question = state['question']
    # form a prompt
    prompt = f'Answer the following question {question}'
    #ask that question to LLM
    answer = model.invoke(prompt).content
    # update the answer in tha state
    state['answer'] = answer
    
    return state

### Creating our graph


In [44]:
graph = StateGraph(LLMState)
## add nodes
graph.add_node('llm_qa',llm_qa)

## add edges
graph.add_edge(START, 'llm_qa')
graph.add_edge('llm_qa', END)

## Compile
workflow = graph.compile()

### Execute

In [47]:
initial_state = {'question': 'How far is moon from the Earth?'}
final_state = workflow.invoke(initial_state)

print(final_state)

{'question': 'How far is moon from the Earth?', 'answer': "The average distance from the Earth to the Moon is about **384,400 kilometers** (approximately 238,855 miles).\n\nThis distance can vary slightly due to the Moon's elliptical orbit around Earth. At its closest point (perigee), the Moon can be as close as about **363,300 kilometers**, and at its farthest point (apogee), it can be as far as about **405,500 kilometers**."}
